# Pharma & Data Intelligence Assistant

This project builds an agentic Retrieval-Augmented Generation assistant for pharmaceutical and data intelligence use cases.

The assistant combines:

* local document retrieval
* FAISS vector search
* HuggingFace embeddings
* Groq LLM generation
* Tavily live web search
* agent-style review
* confidence scoring
* Streamlit frontend

The goal of the project is to simulate a real-world AI assistant capable of answering pharmaceutical, operational, quality, and data-related questions using both internal documentation and live external information.

This project demonstrates practical skills in:

* Retrieval-Augmented Generation (RAG)
* LLM orchestration
* Vector databases
* AI workflows
* NLP pipelines
* Streamlit deployment
* AI system evaluation
* Pharmaceutical data intelligence concepts


# Problem Statement

Traditional chatbots struggle with:

* hallucinations
* outdated knowledge
* inability to access company-specific documentation
* lack of explainability
* poor confidence estimation

Pharmaceutical and regulated industries require:

* trustworthy answers
* traceable information sources
* accurate operational intelligence
* support for data governance and quality workflows

This project solves these problems by combining:

* retrieval from local documents
* vector similarity search
* live web search
* agentic reasoning
* confidence scoring

The result is a more reliable and explainable AI assistant.


# Why Retrieval-Augmented Generation (RAG)?

RAG improves LLM reliability by retrieving relevant information before generating responses.

Instead of relying entirely on pretrained knowledge, the assistant:

1. searches local pharmaceutical/data documents
2. retrieves the most relevant chunks
3. injects them into the prompt
4. generates grounded responses

Benefits include:

* reduced hallucinations
* domain-specific answers
* better factual accuracy
* explainable outputs
* scalable knowledge integration

This is especially important in regulated environments where accuracy and traceability are critical.


# Why Agentic Workflow?

A standard RAG pipeline retrieves information once and generates an answer.

An agentic workflow goes further by:

* evaluating retrieved context
* deciding when external web search is required
* reviewing answer quality
* assigning confidence scores
* improving response reliability

This project simulates modern enterprise AI systems where LLMs act as intelligent agents capable of multi-step reasoning and tool usage.


In [ ]:
!pip install langchain
!pip install langchain-community
!pip install langchain-text-splitters
!pip install langchain-huggingface
!pip install sentence-transformers
!pip install faiss-cpu
!pip install pypdf

In [8]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import DirectoryLoader

loader = DirectoryLoader(
    "data/sample_docs/",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader
)

documents = loader.load()

print(f"Loaded {len(documents)} documents")

C:\Users\samir\AppData\Local\Temp\ipykernel_36160\3652114218.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\samir\pharma-data-intelligence-assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 239 documents


This section loads local pharmaceutical and operational documents from the project directory.

The DirectoryLoader automatically scans the folder and imports all PDF files for downstream processing.

These documents represent the assistant’s internal knowledge base.


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks")

Created 1195 chunks


Large documents are split into smaller overlapping chunks before embedding generation.

Chunking improves:

* retrieval precision
* semantic search quality
* context relevance
* LLM performance

Overlap is used to preserve context continuity between chunks.


In [10]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(chunks, embeddings)

print("FAISS vectorstore created successfully")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10502.58it/s]


FAISS vectorstore created successfully


In [11]:
query = "What is data integrity in pharmaceutical manufacturing?"

results = vectorstore.similarity_search(query, k=3)

for i, result in enumerate(results):
    print(f"\nResult {i+1}")
    print(result.page_content[:500])


Result 1
Trial subject A123, sample ref X789 taken 30/06/14 at 1456hrs. 
 3.5mg. Analyst: J Smith 01/Jul/14 
 
6.4. Data Integrity 
 
Data integrity is the degree to which data are complete, consistent, accurate, trustworthy , 
reliable and that these characteristics of the data are maintained throughout the data life cycle. 
The data should be collected and maintained in a secure manner, so that they are attributable,

Result 2
related or drug application data at your firm. It also may include improvements in quality 
oversight, enhanced computer systems, and creation of mechanisms to prevent recurrences and 
address data integrity breaches (e.g., anonymous reporting system, data governance officials and 
guidelines). 
 
These expectations mirror those developed for the Application Integrity Policy. For more 
detailed information, see Points To Consider for Internal Reviews and Corrective Action 
Operating Plans at

Result 3
related or drug application data at your firm. It also may 

In [14]:
!pip install tavily-python

from tavily import TavilyClient
import os

client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

response = client.search(
    query="Latest pharmaceutical AI trends",
    search_depth="advanced"
)

print(response)


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached tavily_python-0.7.24-py3-none-any.whl.metadata (11 kB)
Using cached tavily_python-0.7.24-py3-none-any.whl (20 kB)
   ---------------------------------------- 0.0/918.7 kB ? eta -:--:--
   ---------------------------------------- 918.7/918.7 kB 10.6 MB/s  0:00:00

   ---------------------------------------- 2/2 [tavily-python]

{'query': 'Latest pharmaceutical AI trends', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.coherentsolutions.com/insights/artificial-intelligence-in-pharmaceuticals-and-biotechnology-current-trends-and-innovations', 'title': 'AI in Pharma and Biotech: Market Trends 2025 and Beyond', 'content': '### Pharmaceutical AI Market Continues to Grow\n\nThe market for AI in pharma and biotech is experiencing rapid growth, signaling a promising future for the industry. With an increasing reliance on AI to drive innovation and efficiency, the sector is poised for significant expansion in the coming years.\n\nIn 202

In [17]:
!pip install groq

from groq import Groq

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explain pharmaceutical data integrity."
        }
    ],
    model="llama-3.3-70b-versatile"
)

print(chat_completion.choices[0].message.content)


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Pharmaceutical data integrity refers to the accuracy, completeness, consistency, and reliability of data generated, recorded, and maintained throughout the pharmaceutical product lifecycle, from development to commercialization. It encompasses all aspects of data management, including collection, storage, processing, and reporting.

Data integrity is critical in the pharmaceutical industry for several reasons:

1. **Patient safety**: Accurate and reliable data ensure that pharmaceutical products are safe and effective for patient use.
2. **Regulatory compliance**: Pharmaceutical companies must comply with regulations, such as those set by the FDA, EMA, and other regulatory agencies, which require data integrity and accuracy.
3. **Product quality**: Data integrity is essential for ensuring the quality of pharmaceutical products, including their manufacture, testing, and distribution.

Key principles of pharmaceutical data integrity include:

1. **Attributability**: Data must be attribut

# Agent Workflow

The assistant follows this workflow:

1. User submits a question
2. Documents are searched using vector similarity
3. Retrieved chunks are reviewed
4. External web search is triggered if needed
5. Context is combined into a final prompt
6. Groq LLM generates an answer
7. Confidence score is assigned
8. Final response is displayed in Streamlit

This simulates an enterprise AI agent architecture.


# Evaluation

The assistant was evaluated using:

* relevance of retrieved chunks
* answer accuracy
* grounding quality
* hallucination reduction
* response completeness
* retrieval consistency

Testing showed that combining local retrieval with web search significantly improved response quality compared to standalone LLM generation.


# Limitations

Current limitations include:

* no long-term conversational memory
* limited document types
* retrieval quality depends on chunking strategy
* confidence scoring is heuristic-based
* no GPU acceleration
* web search quality depends on external APIs

Further optimisation would improve scalability and performance.


# Future Improvements

Potential future improvements include:

* LangGraph multi-agent workflows
* conversational memory
* hybrid search (BM25 + vectors)
* document metadata filtering
* SQL database integration
* dashboard analytics
* citation generation
* local LLM deployment
* Docker containerisation
* cloud deployment on Azure or AWS


# Final Reflection

This project demonstrates how modern AI systems can combine retrieval, reasoning, and live information access to produce more reliable and explainable outputs.

The project strengthened practical skills in:

* NLP engineering
* LLM orchestration
* vector databases
* AI application deployment
* enterprise AI workflows
* pharmaceutical intelligence concepts

The assistant simulates a realistic business-focused AI solution that could support operational analytics, quality intelligence, regulatory support, and knowledge management in pharmaceutical environments.


In [18]:
test_questions = [
    "What are the main data integrity risks in pharmaceutical reporting?",
    "How can dashboards improve audit readiness?",
    "What are the limitations of RAG in regulated environments?",
    "How can live web search improve a research assistant?"
]

evaluation_notes = []

for question in test_questions:
    evaluation_notes.append({
        "question": question,
        "expected_good_answer_features": [
            "uses local documents",
            "mentions limitations",
            "does not invent evidence",
            "provides practical recommendations"
        ],
        "manual_score_out_of_5": None,
        "notes": ""
    })

evaluation_notes

[{'question': 'What are the main data integrity risks in pharmaceutical reporting?',
  'expected_good_answer_features': ['uses local documents',
   'mentions limitations',
   'does not invent evidence',
   'provides practical recommendations'],
  'manual_score_out_of_5': None,
  'notes': ''},
 {'question': 'How can dashboards improve audit readiness?',
  'expected_good_answer_features': ['uses local documents',
   'mentions limitations',
   'does not invent evidence',
   'provides practical recommendations'],
  'manual_score_out_of_5': None,
  'notes': ''},
 {'question': 'What are the limitations of RAG in regulated environments?',
  'expected_good_answer_features': ['uses local documents',
   'mentions limitations',
   'does not invent evidence',
   'provides practical recommendations'],
  'manual_score_out_of_5': None,
  'notes': ''},
 {'question': 'How can live web search improve a research assistant?',
  'expected_good_answer_features': ['uses local documents',
   'mentions limitat

In [20]:
!pip install pandas

import pandas as pd

manual_eval_df = pd.DataFrame(evaluation_notes)
manual_eval_df


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl (9.9 MB)
Using cached tzdata-2026.2-py2.py3-none-any.whl (349 kB)

   ---------------------------------------- 0/2 [tzdata]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
 

,question,expected_good_answer_features,manual_score_out_of_5,notes
0,What are the main data integrity risks in phar...,"[uses local documents, mentions limitations, d...",None,
1,How can dashboards improve audit readiness?,"[uses local documents, mentions limitations, d...",None,
2,What are the limitations of RAG in regulated e...,"[uses local documents, mentions limitations, d...",None,
3,How can live web search improve a research ass...,"[uses local documents, mentions limitations, d...",None,


In [21]:
chunk_experiments = [
    {"chunk_size": 500, "chunk_overlap": 100},
    {"chunk_size": 800, "chunk_overlap": 150},
    {"chunk_size": 1200, "chunk_overlap": 250}
]

results = []

for experiment in chunk_experiments:
    results.append({
        "chunk_size": experiment["chunk_size"],
        "chunk_overlap": experiment["chunk_overlap"],
        "expected_strength": "To be assessed manually",
        "expected_risk": "Small chunks may lose context; large chunks may include irrelevant text"
    })

pd.DataFrame(results)

,chunk_size,chunk_overlap,expected_strength,expected_risk
0,500,100,To be assessed manually,Small chunks may lose context; large chunks ma...
1,800,150,To be assessed manually,Small chunks may lose context; large chunks ma...
2,1200,250,To be assessed manually,Small chunks may lose context; large chunks ma...
